In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/01_silver_patients
# Project  : COVID-19 Clinical Data Lakehouse
# Purpose  : Type casting and data preparation for patients with missing value
#            handling and abbreviation expansion (gender, marital_status)
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.silver.patients",
    comment="Patients with cleaned data types, standardized text values, and expanded code attributes"
)
@dlt.expect("valid_death_after_birth", "death_date IS NULL OR death_date >= birth_date")
def silver_patients():
    df = dlt.read_stream("covid19_lakehouse.bronze.patients")
    
    # 1. Expand marital status abbreviations and handle nulls
    marital_status_expr = (
        F.when(F.lower(F.col("MARITAL")) == "m", F.lit("married"))
         .when(F.lower(F.col("MARITAL")) == "s", F.lit("single"))
         .otherwise(F.coalesce(F.lower(F.col("MARITAL")), F.lit("unknown")))
    )

    # 2. Expand gender abbreviations (m -> male, f -> female) and handle nulls
    gender_expr = (
        F.when(F.lower(F.col("GENDER")) == "m", F.lit("male"))
         .when(F.lower(F.col("GENDER")) == "f", F.lit("female"))
         .otherwise(F.coalesce(F.lower(F.col("GENDER")), F.lit("unknown")))
    )

    return (
        df.select(
            F.col("Id").alias("patient_id"),
            F.col("FIRST").alias("first_name"),
            F.col("LAST").alias("last_name"),
            F.col("PREFIX").alias("prefix"),
            F.col("SUFFIX").alias("suffix"),
            F.col("MAIDEN").alias("maiden_name"),
            F.to_date("BIRTHDATE").alias("birth_date"),
            F.to_date("DEATHDATE").alias("death_date"),
            F.col("SSN").alias("ssn"),
            F.col("DRIVERS").alias("drivers_license"),
            F.col("PASSPORT").alias("passport"),
            gender_expr.alias("gender"),
            F.lower(F.col("RACE")).alias("race"),
            F.lower(F.col("ETHNICITY")).alias("ethnicity"),
            marital_status_expr.alias("marital_status"),
            F.col("BIRTHPLACE").alias("birthplace"),
            F.col("ADDRESS").alias("address"),
            F.col("CITY").alias("city"),
            F.col("STATE").alias("state"),
            F.col("COUNTY").alias("county"),
            F.col("ZIP").alias("zip"),
            F.col("LAT").cast("double").alias("lat"),
            F.col("LON").cast("double").alias("lon"),
            F.col("HEALTHCARE_EXPENSES").cast("decimal(12,2)").alias("healthcare_expenses"),
            F.col("HEALTHCARE_COVERAGE").cast("decimal(12,2)").alias("healthcare_coverage"),
            F.col("_ingested_at"),  # Required for temporal ordering in Gold (SCD2)
        )
    )